In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset, get_dataset_config_names
import random

# ==============================================================================
# 💡 데이터셋 분석 가이드: KOSPI 주식 데이터 탐색 프로젝트
# 💡 이 데이터셋은 2021년~2026년 KOSPI 주식의 일별 시세와 
#    기술적 지표(RSI, MACD 등), 거시 경제 지표(S&P 500, VIX 등)가 
#    결합된 아주 강력한 금광 같은 데이터셋입니다! 💎
# 
# ✨ 목표: 주가 예측 자체는 어렵지만, "오늘의 주가 변동(LogReturn)을 예측하는 데 
#     가장 유용한 지표가 무엇일까?"를 찾아보는 탐색적 데이터 분석(EDA)을 해봅시다!
# ==============================================================================

# --- 설정 변수 ---
DATASET_NAME = "podongchip/kospi-daily-stock-features-2021-2026"
SAMPLE_COUNT = 500  # 코드를 너무 느리게 돌리지 않기 위해 상위 500개 샘플만 사용합니다.

# --- 1. 데이터셋 로드 (스트리밍 최적화) ---

print("✨ [1단계] 데이터셋 로딩 준비: 데이터를 효율적으로 로드해봅시다...")

# 0. 사용 가능한 Config 목록 확인 (필수 패턴)
try:
    configs = get_dataset_config_names(DATASET_NAME)
    if configs:
        print(f"✅ 사용 가능한 Config 목록: {configs}")
        selected_config = configs[0]
    else:
        print("ℹ️ Config 목록을 가져올 수 없습니다. 기본 설정을 사용합니다.")
        selected_config = None
except Exception as e:
    print(f"ℹ️ Config 목록 확인 중 오류 발생: {e}")
    selected_config = None


# 1. 스트리밍 로드를 먼저 시도 (가장 빠르고 메모리 효율적입니다!)
try:
    # split='train'을 명시하고, streaming=True로 설정합니다.
    raw_dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("\n🚀 성공! 스트리밍 모드로 데이터 로딩을 시작합니다. (매우 효율적!)")
    
    # ⚠️ 메모리 및 속도 제약상, 일단 샘플을 가져와서 리스트로 변환합니다.
    # 전체 데이터셋을 list로 로드하는 것은 메모리 초과를 일으킬 수 있습니다.
    sample_data_list = list(raw_dataset.take(SAMPLE_COUNT))

except Exception as e:
    # 😥 만약 스트리밍 모드(IterableDataset)에서 문제가 발생하면,
    # 일반 Dataset으로 강제 다운로드하여 진행합니다.
    print(f"\n⚠️ 스트리밍 로드 실패 ({e}). 일반 Dataset 모드로 전환합니다...")
    try:
        raw_dataset = load_dataset(DATASET_NAME, split='train')
        sample_data_list = list(raw_dataset.take(SAMPLE_COUNT))
        print("✅ 일반 Dataset 모드로 샘플을 성공적으로 로드했습니다. 이제 분석을 시작할게요!")
    except Exception as e_fallback:
        print(f"❌ 데이터셋 로드에 최종적으로 실패했습니다. 오류: {e_fallback}")
        sample_data_list = []

# 2. 데이터프레임으로 변환하여 분석 준비 (Pandas가 분석에 최적입니다)
if sample_data_list:
    # 데이터셋의 각 요소(sample)를 Pandas DataFrame 행으로 처리
    sample_df = pd.DataFrame(sample_data_list)
    print(f"\n🔍 데이터 전처리 완료: {sample_df.shape[0]}개의 샘플 데이터를 분석할 준비를 마쳤습니다.")
else:
    sample_df = pd.DataFrame()
    print("🛑 데이터프레임 생성에 실패하여 분석을 종료합니다.")


# ==============================================================================
# 💰 [2단계] 창의적 분석 실습: 오늘 주가 상승에 결정적인 영향을 준 요인은 무엇일까?
# ==============================================================================
if not sample_df.empty:
    
    print("\n\n=================================================================")
    print("🚀 [2단계] AI 탐색자 모드: 주가 변동과 지표 간의 상관관계 분석 📊")
    print("=================================================================")
    
    # 🌟 목표 지표 정의: 우리가 예측하고 싶은 것은 '일별 수익률 (LogReturn)'입니다.
    # 🌟 분석 대상 지표 정의: LogReturn에 영향을 줄 만한 외부 요인들을 골라봅시다.
    
    # A. 핵심 거시 지표 (Macro Indicators) 선택: 글로벌 시장의 분위기가 어땠는지?
    macro_cols = [
        'KOSPI_ret',    # 국내 시장 수익률 (자기 자신과의 상관관계를 볼 수 있습니다)
        'SP500_ret',    # 미국 S&P 500 수익률 (해외 영향력 측정!)
        'VIX_ret'       # 공포 지수 수익률 (수치가 높으면 시장이 불안했다는 의미)
    ]
    
    # B. 핵심 기술 지표 (Technical Indicators) 선택: 시장 참여자들이 주로 보는 건 무엇일까?
    tech_cols = [
        'RSI14',        # 상대강도지수: 과매수/과매도 상태를 알려줍니다.
        'MACD'          # MACD: 추세의 변화를 잡아줍니다.
    ]
    
    # 🚀 분석할 컬럼들을 통합합니다.
    analysis_cols = list(set(macro_cols + tech_cols))
    
    # 데이터프레임에 분석할 컬럼들만 추출합니다. (결측치 제거를 위해 필요한 작업)
    analysis_df = sample_df[analysis_cols].dropna()
    
    if analysis_df.empty:
        print("🚨 분석에 필요한 지표들이 샘플에 충분히 존재하지 않거나 결측치가 너무 많습니다. 분석을 건너뜁니다.")
    else:
        print(f"\n[상위 {analysis_df.shape[0]}개 샘플에 대해 상관관계를 계산합니다...")
        print("👉 분석 대상 컬럼: " + ", ".join(analysis_cols))
        
        # 📈 상관관계 분석 실행: 이 지표들들이 얼마나 함께 움직이는지 숫자로 확인합니다.
        correlation_matrix = analysis_df.corr()
        
        # 📊 핵심 지표들만 추출하여 보기 쉽게 만듭니다.
        target_corr = correlation_matrix['LogReturn']
        
        print("\n=================================================================")
        print("💡 🔍 오늘의 주가 움직임(LogReturn)에 대한 주요 지표별 상관계수:")
        print("=================================================================")
        
        # Pandas의 to_string()을 사용해 출력 포맷을 깔끔하게 만듭니다.
        print(target_corr.sort_values(ascending=False).to_string())
        
        print("\n[튜터의 위트 있는 해석 😉]")
        print("   - 상관계수 값이 +1에 가깝다면: 해당 지표가 높을 때 주가도 함께 오르는 경향이 있습니다. (긍정적 연관)")
        print("   - 상관계수 값이 -1에 가깝다면: 해당 지표가 높을 때 주가가 내리는 경향이 있습니다. (역방향 연관)")
        print("   - 값이 0에 가깝다면: 거의 연관성이 없다는 뜻입니다. (독립적)")
        
        # 🖼️ 시각화 (선택적: Matplotlib을 사용해 상위 3개만 빠르게 시각화)
        
        # 상관관계가 높은 상위 3개 지표를 선택합니다.
        top_corr_features = target_corr.abs().sort_values(ascending=False).index[:3]
        plot_data = analysis_df[top_corr_features]
        
        plt.figure(figsize=(10, 6))
        
        # Scatter plot 대신, 시계열 데이터의 추세를 보여주기 위해 pd.plotting.scatter_plot는 사용하지 않고 
        # 간단히 로그 수익률과 가장 상관관계 높은 변수를 함께 찍는 산점도를 보여줍니다.
        for feature in top_corr_features:
            plt.subplot(1, 3, top_corr_features.tolist().index(feature) + 1)
            plt.scatter(analysis_df[feature].dropna(), analysis_df['LogReturn'].dropna(), alpha=0.5)
            plt.title(f'{feature} vs LogReturn')
            plt.xlabel(f'{feature} Value')
            plt.ylabel('LogReturn')
        
        plt.tight_layout()
        plt.show()
        
        print("\n\n✨ 축하합니다! 기본적인 통계적 탐색을 완료하셨어요!")
        print("이처럼 데이터를 파헤치는 과정이 AI 모델링의 첫걸음입니다. 다음 단계는 이 패턴을 기반으로 실제로 '오늘의 주가'를 예측하는 모델을 만들어보는 것이랍니다! 💪")